# Stage 5: Lightweight Interpretable Model Comparison

This controlled comparison includes the established weekday baseline, the Stage 4 Ridge check, and one fixed HistGradientBoosting model. There is no tuning search, cross-validation, ensemble, advanced interpretation package, or Kaggle test forecast.

The script built the Stage 4 matrices once, encoded once, and fitted Ridge and HistGradientBoosting once each. This notebook reads the generated results and figures without refitting.

In [1]:
from pathlib import Path
import pandas as pd

def find_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "reports" / "tables" / "model_scores.csv").exists():
            return candidate
    raise FileNotFoundError("Run python -m src.models first")

ROOT = find_root()
scores = pd.read_csv(ROOT / "reports" / "tables" / "model_scores.csv")
errors = pd.read_csv(ROOT / "reports" / "tables" / "model_error_breakdown.csv")
print("Validation: 2017-07-31 to 2017-08-15 (16 days, 28,512 rows)")
print("Shared Stage 4 inputs: 29 encoded features; 648,648 training rows")
display(scores)

Validation: 2017-07-31 to 2017-08-15 (16 days, 28,512 rows)
Shared Stage 4 inputs: 29 encoded features; 648,648 training rows


,rank,model_name,model_type,validation_rmsle,difference_from_weekday_baseline,difference_from_ridge,input_feature_count,training_window,training_time_seconds,specification
0,1,HistGradientBoosting,nonlinear histogram boosting,0.449788,-0.070843,-0.045150,29,final 365 days before validation,3.529074,"learning_rate=0.08, max_iter=120, max_leaf_nod..."
1,2,Ridge regression,regularized linear,0.494938,-0.025693,0.000000,29,final 365 days before validation,0.101151,"Ridge(alpha=1.0), log1p target"
2,3,Weekday mean baseline,seasonal baseline,0.520631,0.000000,0.025693,0,previous 8 weeks,0.000000,store-family weekday mean


## 1. Shared validation and features

Both fitted models use the same final-365-day training matrix and validation keys. The Stage 4 recommendation is unchanged: calendar, store type/cluster, promotion, lags 7/14/28, rolling summaries 7/28, rolling standard deviation 28, and existing log-history transformations. Holiday and oil remain excluded. The target perturbation leakage check passed before fitting.

## 2. Models and computational controls

- **Weekday mean:** Stage 3 score reused without rerunning the baseline sweep.
- **Ridge:** alpha 1.0 and a `log1p` target.
- **HistGradientBoosting:** learning rate 0.08, up to 120 iterations, 31 leaves, early stopping, random state 42.

Features and preprocessing are shared rather than rebuilt per model.

![Overall model ranking](../reports/figures/models/01_model_ranking.png)

## 3. Limited error analysis

The compact outputs identify the highest-error families, stores, and dates without plotting every group.

![Family RMSLE comparison](../reports/figures/models/02_family_errors.png)

![Validation-date RMSLE comparison](../reports/figures/models/03_date_errors.png)

In [2]:
for group_type in ["family", "store", "date"]:
    print(f"Highest {group_type} errors")
    subset = errors.loc[errors["group_type"] == group_type]
    display(subset.sort_values(["model_name", "rmsle"], ascending=[True, False]).groupby("model_name").head(5))

Highest family errors


,group_type,group_value,model_name,rmsle
134,family,SCHOOL AND OFFICE SUPPLIES,HistGradientBoosting,1.133558
116,family,GROCERY II,HistGradientBoosting,0.700822
124,family,LINGERIE,HistGradientBoosting,0.646195
109,family,CELEBRATION,HistGradientBoosting,0.571717
126,family,MAGAZINES,HistGradientBoosting,0.548606
31,family,SCHOOL AND OFFICE SUPPLIES,Ridge regression,1.455789
13,family,GROCERY II,Ridge regression,0.711224
21,family,LINGERIE,Ridge regression,0.671073
22,family,"LIQUOR,WINE,BEER",Ridge regression,0.600554
6,family,CELEBRATION,Ridge regression,0.578347


Highest store errors


,group_type,group_value,model_name,rmsle
185,store,50,HistGradientBoosting,0.695486
182,store,47,HistGradientBoosting,0.680710
179,store,44,HistGradientBoosting,0.638349
183,store,48,HistGradientBoosting,0.628402
154,store,19,HistGradientBoosting,0.625712
82,store,50,Ridge regression,0.870598
79,store,47,Ridge regression,0.846397
80,store,48,Ridge regression,0.764128
76,store,44,Ridge regression,0.726770
51,store,19,Ridge regression,0.645162


Highest date errors


,group_type,group_value,model_name,rmsle
201,date,2017-08-11 00:00:00,HistGradientBoosting,0.511398
205,date,2017-08-15 00:00:00,HistGradientBoosting,0.498996
203,date,2017-08-13 00:00:00,HistGradientBoosting,0.493846
202,date,2017-08-12 00:00:00,HistGradientBoosting,0.490205
204,date,2017-08-14 00:00:00,HistGradientBoosting,0.472028
98,date,2017-08-11 00:00:00,Ridge regression,0.544470
100,date,2017-08-13 00:00:00,Ridge regression,0.533095
99,date,2017-08-12 00:00:00,Ridge regression,0.531889
101,date,2017-08-14 00:00:00,Ridge regression,0.524760
88,date,2017-08-01 00:00:00,Ridge regression,0.520141


## 4. Representative forecasts

The low-, medium-, and high-volume examples follow the earlier mechanical selection rule. Both models can still miss intermittent changes and spikes.

![Representative validation forecasts](../reports/figures/models/04_representative_forecasts.png)

## 5. Preferred model for Stage 6

HistGradientBoosting is preferred because its validation improvement over Ridge is clear and its runtime remains modest. This is predictive comparison rather than causal evidence, and validation leadership does not guarantee Kaggle leaderboard leadership. Stage 6 has not begun.